In [0]:
import pandas as pd
from pyspark.sql.functions import lit, current_timestamp

# 1. Configuration
las_path = "/Volumes/equinor_asa_northern_lights/public/northernlights/31_5-7 Eos/05.LWD_Log_data/WL_RAW_BHPR-GR-MECH_TIME_MWD_1.LAS"
table_name = "crude_ops.bronze.drilling_raw"

try:
    # 2. Extraction
    with open(las_path, "rb") as f:
        content = f.read().decode('latin-1', errors='ignore')

    lines = content.splitlines()
    columns = []
    data_rows = []
    in_curve = False
    in_data = False

    for line in lines:
        clean_line = line.strip()
        if not clean_line: continue
        
        if clean_line.upper().startswith("~C"):
            in_curve, in_data = True, False
            continue
        if clean_line.upper().startswith("~A"):
            in_data, in_curve = True, False
            continue
        
        if in_curve and not clean_line.startswith("#"):
            col_name = clean_line.split(".")[0].strip()
            columns.append(col_name)
            
        if in_data and not clean_line.startswith("~") and not clean_line.startswith("#"):
            row_values = clean_line.split()
            if len(row_values) == len(columns):
                data_rows.append(row_values)

    # 3. Processing to Spark
    pdf = pd.DataFrame(data_rows, columns=columns).apply(pd.to_numeric, errors='coerce')
    bronze_drilling_df = spark.createDataFrame(pdf)

    # Add audit metadata (Best practice for Principal DE)
    bronze_drilling_df = bronze_drilling_df.withColumn("ingestion_timestamp", current_timestamp()) \
                                           .withColumn("source_file", lit(las_path))

    # 4. Load to Unity Catalog
    bronze_drilling_df.write.format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .saveAsTable(table_name)

    print(f"SUCCESS: {bronze_drilling_df.count()} rows saved to {table_name}")

except Exception as e:
    print(f"ERROR: {str(e)}")

In [0]:
%skip
%sql
select * from crude_ops.bronze.drilling_raw